# Team C — C3 Ottimizzato (EstimatorV2 batched + SPSA gradiente)

Ottimizzazioni applicate rispetto al notebook base:
- `AerEstimator` (EstimatorV2) al posto di `StatevectorEstimator` → batching nativo
- SPSA usato come **stima del gradiente nel backward**, non come ottimizzatore esterno
- Subsample del training set (2000 sample) per rendere il training fattibile
- Readout progressivo: `z` → `x_y_z` → `x_y_z_pair_nn`
- Adam sulla testa classica, SPSA solo sui pesi quantistici

## 1. Import

In [48]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import math, os, json
from time import perf_counter

from sklearn.metrics import roc_auc_score, f1_score, balanced_accuracy_score
from torchmetrics.classification import MulticlassCalibrationError

from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

# ─── Costanti condivise ────────────────────────────────────────────────────────
CURRENT_SEED = 17
COMPONENTS   = [32, 16, 8, 4]
N_QUBITS     = 8

os.makedirs('../artifacts', exist_ok=True)
print('OK')

OK


## 2. Configurazione ottimizzata

Tutti i parametri in un unico posto. Modifica qui prima di lanciare.

In [ ]:
# ─── Configurazione (dal PDF) ──────────────────────────────────────────────────
CFG = dict(
    # Backend
    backend          = 'aer-statevector',   # AerEstimator con statevector
    estimator_shots  = None,                # None = exact (no sampling noise)

    # Readout — parti da 'z', poi aumenta
    # Opzioni: 'z' | 'x_y_z' | 'x_y_z_pair_nn' | 'x_y_z_pair_all'
    readout_selector = 'z',

    # Ansatz
    reps             = 1,
    topology         = 'ring',              # 'ring' o 'ladder'

    # SPSA come gradiente
    spsa_epsilon     = 1e-3,                # ampiezza perturbazione
    spsa_batch_size  = 1,                   # campioni SPSA per stima gradiente

    # Training
    batch_size       = 4,                   # piu alto -> veloce
    n_epochs         = 100,
    patience         = 20,
    lr_classical     = 1e-3,                # Adam per la testa lineare
    lr_quantum       = 0.05,                # learning rate SPSA pesi quantistici

    # Subsample — fondamentale con 97k sample
    n_train_samples  = 500,                # None = usa tutto
)

# Dimensione output in base al readout
READOUT_DIM = {
    'z':              N_QUBITS,                      #  8
    'x_y_z':          N_QUBITS * 3,                  # 24
    'x_y_z_pair_nn':  N_QUBITS * 3 + (N_QUBITS - 1), # 45 (con 8 qubit)
    'x_y_z_pair_all': N_QUBITS * 3 + N_QUBITS * (N_QUBITS - 1) // 2,  # 108
}

OUT_DIM = READOUT_DIM[CFG['readout_selector']]
print(f"readout={CFG['readout_selector']} → output_dim={OUT_DIM}")

readout=z → output_dim=8


## 3. Costruzione osservabili (readout progressivo)

In [50]:
def build_observables_readout(n_qubits, readout_selector):
    """
    Costruisce la lista di osservabili SparsePauliOp in base al readout scelto.

    z             → ⟨Z_i⟩ per ogni qubit                         (8 obs)
    x_y_z         → ⟨X_i⟩, ⟨Y_i⟩, ⟨Z_i⟩ per ogni qubit         (24 obs)
    x_y_z_pair_nn → x_y_z + ⟨Z_i Z_{i+1}⟩ vicini               (45 obs)
    x_y_z_pair_all→ x_y_z + ⟨Z_i Z_j⟩ tutte le coppie          (108 obs)
    """
    obs = []

    def pauli_single(op, qubit, n):
        """Es: pauli_single('Z', 2, 8) → 'IIIIIZII'"""
        s = ['I'] * n
        s[n - qubit - 1] = op
        return SparsePauliOp(''.join(s))

    def pauli_pair(op1, op2, q1, q2, n):
        """Es: ZZ su qubit 0 e 1 → 'IIIIIZZ'"""
        s = ['I'] * n
        s[n - q1 - 1] = op1
        s[n - q2 - 1] = op2
        return SparsePauliOp(''.join(s))

    # ── Z su tutti i qubit (sempre presente) ──────────────────────────────────
    for i in range(n_qubits):
        obs.append(pauli_single('Z', i, n_qubits))

    if readout_selector in ('x_y_z', 'x_y_z_pair_nn', 'x_y_z_pair_all'):
        # ── X e Y su tutti i qubit ─────────────────────────────────────────────
        for i in range(n_qubits):
            obs.append(pauli_single('X', i, n_qubits))
        for i in range(n_qubits):
            obs.append(pauli_single('Y', i, n_qubits))

    if readout_selector == 'x_y_z_pair_nn':
        # ── ZZ su coppie adiacenti (nearest-neighbour) ─────────────────────────
        for i in range(n_qubits - 1):
            obs.append(pauli_pair('Z', 'Z', i, i + 1, n_qubits))

    if readout_selector == 'x_y_z_pair_all':
        # ── ZZ su tutte le coppie ──────────────────────────────────────────────
        for i in range(n_qubits):
            for j in range(i + 1, n_qubits):
                obs.append(pauli_pair('Z', 'Z', i, j, n_qubits))

    return obs


# Verifica
for sel in ['z', 'x_y_z', 'x_y_z_pair_nn', 'x_y_z_pair_all']:
    obs = build_observables_readout(N_QUBITS, sel)
    print(f'{sel:20s} → {len(obs):3d} osservabili')

z                    →   8 osservabili
x_y_z                →  24 osservabili
x_y_z_pair_nn        →  31 osservabili
x_y_z_pair_all       →  52 osservabili


## 4. Encoding e ansatz C3 (identici al baseline)

In [51]:
def build_encoding_layer(d, n_qubits, block_idx):
    params = ParameterVector(f'x_{block_idx}', n_qubits)
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(params[i], i)
    return qc, params


def build_topology_ansatz(n_qubits, topology, block_idx, reps=1):
    n_params   = n_qubits * (reps + 1)
    var_params = ParameterVector(f'θ_{block_idx}', n_params)
    param_idx  = 0
    qc = QuantumCircuit(n_qubits)

    for _ in range(reps):
        for i in range(n_qubits):
            qc.ry(var_params[param_idx], i)
            param_idx += 1
        if topology == 'ring':
            for i in range(n_qubits):
                qc.cx(i, (i + 1) % n_qubits)
        elif topology == 'ladder':
            for i in range(0, n_qubits - 1, 2):
                qc.cx(i, i + 1)
            for i in range(1, n_qubits - 1, 2):
                qc.cx(i, i + 1)
        else:
            raise ValueError(f"Topologia non supportata: {topology}")

    for i in range(n_qubits):
        qc.ry(var_params[param_idx], i)
        param_idx += 1

    return qc, list(var_params)


def build_vqc_circuit_c3(d, n_qubits=N_QUBITS, reps=1, topology='ring'):
    n_blocks = math.ceil(d / n_qubits)
    qc = QuantumCircuit(n_qubits)
    all_input_params, all_var_params = [], []

    for block in range(n_blocks):
        enc_layer, enc_params = build_encoding_layer(d, n_qubits, block)
        qc.compose(enc_layer, inplace=True)
        all_input_params.extend(enc_params)

        ansatz, var_params = build_topology_ansatz(n_qubits, topology, block, reps)
        qc.compose(ansatz, inplace=True)
        all_var_params.extend(var_params)

    return qc, all_input_params, all_var_params, n_blocks


print('Encoding e ansatz definiti.')

Encoding e ansatz definiti.


## 5. QNN con AerEstimator (EstimatorV2 batched)

In [52]:
def build_aer_estimator():
    """
    AerEstimator (EstimatorV2) con statevector esatto.
    Supporta batching nativo: calcola tutte le aspettazioni
    del batch in una sola chiamata invece di N chiamate separate.
    """
    estimator = AerEstimator()
    estimator.options.shots = None          # exact, no sampling noise
    estimator.options.method = 'statevector'
    return estimator


def build_qnn_c3_optimized(d, n_qubits=N_QUBITS, topology='ring', reps=1, readout_selector='z'):
    """
    QNN ottimizzato:
    - AerEstimator (batched) invece di StatevectorEstimator
    - Readout configurabile (z / x_y_z / x_y_z_pair_nn)
    - Nessun gradiente analitico (ParamShift) → il gradiente
      viene stimato da SPSA nel training loop
    """
    qc, input_params, var_params, _ = build_vqc_circuit_c3(d, n_qubits, reps, topology)
    observables = build_observables_readout(n_qubits, readout_selector)
    estimator   = build_aer_estimator()

    qnn = EstimatorQNN(
        circuit=qc,
        observables=observables,
        input_params=input_params,
        weight_params=var_params,
        estimator=estimator,
        input_gradients=False   # gradiente input non serve, solo pesi
    )

    out_dim = len(observables)
    print(f'd={d:2d} | topology={topology} | readout={readout_selector} | '
          f'output_dim={out_dim} | n_weights={len(var_params)}')
    return qnn, out_dim


# Test rapido
qnn_test, out_dim_test = build_qnn_c3_optimized(
    d=4, topology=CFG['topology'],
    readout_selector=CFG['readout_selector']
)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


d= 4 | topology=ring | readout=z | output_dim=8 | n_weights=16


## 6. Modello ibrido con testa adattiva

In [53]:
def build_hybrid_model_optimized(d, n_qubits=N_QUBITS, topology='ring', reps=1, readout_selector='z'):
    """
    Modello ibrido ottimizzato.
    La testa classica si adatta automaticamente alla dimensione del readout:
      z             → Linear(8,  4)
      x_y_z         → Linear(24, 4)
      x_y_z_pair_nn → Linear(45, 4)
    """
    qnn, out_dim = build_qnn_c3_optimized(d, n_qubits, topology, reps, readout_selector)
    model = nn.Sequential(
        TorchConnector(qnn),
        nn.Linear(out_dim, 4)
    )
    return model, out_dim


def init_weights(model, seed, out_dim):
    torch.manual_seed(seed)
    for name, param in model.named_parameters():
        if 'weight' in name and param.shape == torch.Size([4, out_dim]):
            nn.init.xavier_uniform_(param)
        else:
            nn.init.uniform_(param, -0.1, 0.1)
    return model


def apply_padding(X, d, n_qubits):
    n_blocks    = math.ceil(d / n_qubits)
    padded_size = n_blocks * n_qubits
    if padded_size == d:
        return X
    return torch.cat([X, torch.zeros(X.shape[0], padded_size - d)], dim=1)


print('Modello definito.')

Modello definito.


## 7. SPSA come gradiente (non ottimizzatore esterno)

Differenza chiave rispetto al baseline:
- **Prima**: SPSA faceva 2 forward pass per aggiornare i parametri direttamente
- **Ora**: SPSA stima il gradiente, poi Adam lo usa per aggiornare → separazione classico/quantistico

In [54]:
def get_quantum_params(model):
    """Restituisce solo i parametri del QNN (TorchConnector), non la testa classica."""
    return [p for name, p in model.named_parameters() if 'weight' not in name or p.dim() == 1]


def get_classical_params(model, out_dim):
    """Restituisce solo i parametri della testa classica Linear(out_dim, 4)."""
    return [p for name, p in model.named_parameters()
            if 'weight' in name and p.shape == torch.Size([4, out_dim])]


def spsa_gradient_step(model, X_batch, y_batch, loss_fn,
                        quantum_params, epsilon=1e-3):
    """
    Stima il gradiente dei soli pesi quantistici con SPSA e lo scrive
    in param.grad, in modo che l'ottimizzatore esterno (Adam) possa usarlo.

    Differenza dal baseline:
    - Non aggiorna i parametri direttamente
    - Scrive solo .grad → poi sarà optimizer.step() a fare l'update
    """
    original = [p.data.clone() for p in quantum_params]
    deltas   = [torch.randint(0, 2, p.shape).float() * 2 - 1 for p in quantum_params]

    with torch.no_grad():
        # Forward pass +epsilon
        for p, d in zip(quantum_params, deltas):
            p.data += epsilon * d
        loss_plus = loss_fn(model(X_batch), y_batch)

        # Forward pass -epsilon
        for p, d, orig in zip(quantum_params, deltas, original):
            p.data = orig - epsilon * d
        loss_minus = loss_fn(model(X_batch), y_batch)

        # Ripristina parametri originali
        for p, orig in zip(quantum_params, original):
            p.data = orig.clone()

        # Scrivi gradiente stimato in .grad
        for p, d in zip(quantum_params, deltas):
            grad = (loss_plus - loss_minus) / (2 * epsilon * d)
            if p.grad is None:
                p.grad = grad.clone()
            else:
                p.grad.copy_(grad)

    return ((loss_plus + loss_minus) / 2).item()


print('SPSA gradiente definito.')

SPSA gradiente definito.


## 8. Benchmark singolo batch (esegui SEMPRE prima del training)

In [55]:
def benchmark_batch_real(d, cfg, n_trials=5):
    ckpt    = torch.load(f'../compressedFeatures/angular_d{d}_seed_{CURRENT_SEED}.pt', weights_only=False)
    X_batch = apply_padding(ckpt['train_ang'][:cfg['batch_size']], d, N_QUBITS)
    y_batch = ckpt['y_train'].squeeze().long()[:cfg['batch_size']]
    loss_fn = nn.CrossEntropyLoss()

    model, out_dim = build_hybrid_model_optimized(
        d, N_QUBITS, cfg['topology'], cfg['reps'], cfg['readout_selector']
    )
    q_params = [p for p in model[0].parameters()]

    # warmup
    spsa_gradient_step(model, X_batch, y_batch, loss_fn, q_params, cfg['spsa_epsilon'])

    # misura reale
    start = perf_counter()
    for _ in range(n_trials):
        spsa_gradient_step(model, X_batch, y_batch, loss_fn, q_params, cfg['spsa_epsilon'])
    elapsed = (perf_counter() - start) / n_trials

    print(f'd={d} | tempo medio per batch={elapsed:.2f}s')
    print(f'Stima per epoca (4000 campioni): {elapsed * 1000:.0f}s = {elapsed * 1000 / 60:.1f} min')
    return elapsed

benchmark_batch_real(4, CFG)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


d= 4 | topology=ring | readout=z | output_dim=8 | n_weights=16
d=4 | tempo medio per batch=0.03s
Stima per epoca (4000 campioni): 31s = 0.5 min


0.030647848599619464

## 9. Training ottimizzato

In [56]:
def train_vqc_c3_optimized(d, cfg, seed=CURRENT_SEED):
    """
    Training ottimizzato per C3:
    - Subsample del training set
    - SPSA solo sui pesi quantistici
    - Adam sulla testa classica
    - AerEstimator batched
    """
    # ── Carica dati ───────────────────────────────────────────────────────────
    ckpt    = torch.load(f'../compressedFeatures/angular_d{d}_seed_{seed}.pt', weights_only=False)
    X_train = apply_padding(ckpt['train_ang'], d, N_QUBITS)
    X_val   = apply_padding(ckpt['val_ang'],   d, N_QUBITS)
    y_train = ckpt['y_train'].squeeze().long()
    y_val   = ckpt['y_val'].squeeze().long()

    # ── Subsample ─────────────────────────────────────────────────────────────
    if cfg['n_train_samples'] is not None:
        torch.manual_seed(seed)
        idx     = torch.randperm(len(X_train))[:cfg['n_train_samples']]
        X_train = X_train[idx]
        y_train = y_train[idx]
        print(f'Subsample: {len(X_train)} sample train')

    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=cfg['batch_size'], shuffle=True
    )

    # ── Modello ───────────────────────────────────────────────────────────────
    model, out_dim = build_hybrid_model_optimized(
        d, N_QUBITS, cfg['topology'], cfg['reps'], cfg['readout_selector']
    )
    model = init_weights(model, seed, out_dim)

    # ── Ottimizzatori separati ────────────────────────────────────────────────
    # Parametri quantistici: aggiornati con SPSA (scritto in .grad)
    # Parametri classici: aggiornati con Adam
    quantum_params   = list(model[0].parameters())   # TorchConnector
    classical_params = list(model[1].parameters())   # Linear

    optimizer = torch.optim.Adam(classical_params, lr=cfg['lr_classical'])
    loss_fn   = nn.CrossEntropyLoss()

    # ── Training loop ─────────────────────────────────────────────────────────
    best_val_loss    = float('inf')
    patience_counter = 0
    best_weights     = None
    history          = {'train_loss': [], 'val_loss': []}

    print(f'\nd={d} | topology={cfg["topology"]} | readout={cfg["readout_selector"]} | '
          f'batch={cfg["batch_size"]} | epochs={cfg["n_epochs"]} | patience={cfg["patience"]}')
    print('-' * 70)

    for epoch in range(cfg['n_epochs']):
        model.train()
        epoch_losses = []

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()

            # 1. Stima gradiente pesi quantistici con SPSA
            loss_val = spsa_gradient_step(
                model, X_batch, y_batch, loss_fn,
                quantum_params, cfg['spsa_epsilon']
            )

            # 2. Forward per la testa classica (con autograd)
            logits = model(X_batch)
            loss   = loss_fn(logits, y_batch)
            loss.backward()      # gradiente per la testa classica

            # 3. Aggiornamento pesi quantistici (manuale, con lr dedicato)
            with torch.no_grad():
                for p in quantum_params:
                    if p.grad is not None:
                        p.data -= cfg['lr_quantum'] * p.grad

            # 4. Aggiornamento testa classica con Adam
            optimizer.step()

            epoch_losses.append(loss_val)

        train_loss = np.mean(epoch_losses)

        # ── Validation ────────────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val), y_val).item()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        print(f'epoch {epoch+1:3d}/{cfg["n_epochs"]} | train={train_loss:.4f} | val={val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            patience_counter = 0
            best_weights     = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= cfg['patience']:
                print(f'Early stopping a epoch {epoch+1}')
                break

    model.load_state_dict(best_weights)
    print(f'Miglior val_loss: {best_val_loss:.4f}')
    return model, history, out_dim

## 10. Evaluation e salvataggio artifact

In [57]:
def evaluate_vqc(model, X, y, n_classes=4):
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(X), dim=1).numpy()
        preds = probs.argmax(axis=1)
    y_np  = y.numpy()
    auroc = roc_auc_score(y_np, probs, multi_class='ovr', average='macro')
    f1    = f1_score(y_np, preds, average='macro', zero_division=0)
    bacc  = balanced_accuracy_score(y_np, preds)
    ece   = MulticlassCalibrationError(num_classes=n_classes, n_bins=15, norm='l1')(
        torch.tensor(probs, dtype=torch.float32),
        torch.tensor(y_np,  dtype=torch.long)
    ).item()
    return {'macro_auroc': round(auroc,4), 'macro_f1': round(f1,4),
            'bal_acc': round(bacc,4), 'ece': round(ece,4)}, preds, probs


def evaluate_and_save_c3(model, d, history, topology, readout_selector,
                          seed=CURRENT_SEED, n_qubits=N_QUBITS):
    ckpt   = torch.load(f'../compressedFeatures/angular_d{d}_seed_{seed}.pt', weights_only=False)
    X_val  = apply_padding(ckpt['val_ang'],  d, n_qubits)
    X_test = apply_padding(ckpt['test_ang'], d, n_qubits)
    y_val  = ckpt['y_val'].squeeze().long()
    y_test = ckpt['y_test'].squeeze().long()

    val_m,  val_preds,  val_probs  = evaluate_vqc(model, X_val,  y_val)
    test_m, test_preds, test_probs = evaluate_vqc(model, X_test, y_test)

    print(f'd={d:2d} | topology={topology} | readout={readout_selector}')
    print(f'  VAL  → AUROC={val_m["macro_auroc"]:.4f} | F1={val_m["macro_f1"]:.4f} | BalAcc={val_m["bal_acc"]:.4f} | ECE={val_m["ece"]:.4f}')
    print(f'  TEST → AUROC={test_m["macro_auroc"]:.4f} | F1={test_m["macro_f1"]:.4f} | BalAcc={test_m["bal_acc"]:.4f} | ECE={test_m["ece"]:.4f}')

    ansatz_name = f'TopologyAware_{topology.capitalize()}_C3_opt'
    artifact = {
        'd': d, 'seed': seed, 'model': 'VQC', 'ansatz': ansatz_name,
        'topology': topology, 'readout': readout_selector,
        'n_qubits': n_qubits, 'compression': 'pca_baseline', 'type': 'quantum_C3_optimized',
        'val_metrics': val_m, 'test_metrics': test_m
    }
    json_path = f'../artifacts/vqc_{ansatz_name}_d{d}_seed_{seed}.json'
    with open(json_path, 'w') as f:
        json.dump(artifact, f, indent=2)
    np.savez(
        f'../artifacts/vqc_{ansatz_name}_d{d}_seed_{seed}_preds.npz',
        val_y_true=y_val.numpy(), val_y_pred=val_preds, val_probs=val_probs,
        test_y_true=y_test.numpy(), test_y_pred=test_preds, test_probs=test_probs,
        train_loss=np.array(history['train_loss']), val_loss=np.array(history['val_loss'])
    )
    print(f'Salvato: {json_path}')

## 11. Sequenza di test (dal PDF)

Esegui in ordine. Passa al prossimo step solo se il benchmark è < 30s.

In [41]:
import torch

d = 4
seed = 17

ckpt = torch.load(f'../compressedFeatures/angular_d{d}_seed_{seed}.pt', weights_only=False)

print("Chiavi nel file:", ckpt.keys())
print()
print(f"train_ang shape: {ckpt['train_ang'].shape}")
print(f"val_ang shape:   {ckpt['val_ang'].shape}")
print(f"test_ang shape:  {ckpt['test_ang'].shape}")
print()
print(f"range train: [{ckpt['train_ang'].min():.4f}, {ckpt['train_ang'].max():.4f}]")
print(f"range val:   [{ckpt['val_ang'].min():.4f}, {ckpt['val_ang'].max():.4f}]")
print(f"range test:  [{ckpt['test_ang'].min():.4f}, {ckpt['test_ang'].max():.4f}]")

Chiavi nel file: dict_keys(['train_ang', 'val_ang', 'test_ang', 'y_train', 'y_val', 'y_test', 'd', 'seed', 'angular_scaling'])

train_ang shape: torch.Size([4000, 4])
val_ang shape:   torch.Size([1000, 4])
test_ang shape:  torch.Size([1000, 4])

range train: [0.0000, 6.2832]
range val:   [0.0000, 6.2832]
range test:  [0.0394, 5.0199]


In [42]:
# ── STEP 1: d=4, readout=z ────────────────────────────────────────────────────
CFG['readout_selector'] = 'z'
CFG['batch_size']       = 4

model_d4, history_d4, out_dim_d4 = train_vqc_c3_optimized(d=4, cfg=CFG)
evaluate_and_save_c3(model_d4, d=4, history=history_d4,
                      topology=CFG['topology'], readout_selector=CFG['readout_selector'])

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


Subsample: 500 sample train
d= 4 | topology=ring | readout=z | output_dim=8 | n_weights=16

d=4 | topology=ring | readout=z | batch=4 | epochs=100 | patience=20
----------------------------------------------------------------------
epoch   1/100 | train=1.4417 | val=1.4449
epoch   2/100 | train=1.4759 | val=1.5613
epoch   3/100 | train=1.4612 | val=1.4623
epoch   4/100 | train=1.4338 | val=1.4562
epoch   5/100 | train=1.4613 | val=1.3868
epoch   6/100 | train=1.4192 | val=1.3712
epoch   7/100 | train=1.4123 | val=1.3608
epoch   8/100 | train=1.4090 | val=1.4553
epoch   9/100 | train=1.4257 | val=1.4015
epoch  10/100 | train=1.4088 | val=1.3688
epoch  11/100 | train=1.3982 | val=1.4142
epoch  12/100 | train=1.4217 | val=1.4407
epoch  13/100 | train=1.4238 | val=1.4102
epoch  14/100 | train=1.4038 | val=1.3897
epoch  15/100 | train=1.3734 | val=1.3844
epoch  16/100 | train=1.3886 | val=1.3947
epoch  17/100 | train=1.3938 | val=1.3918
epoch  18/100 | train=1.3836 | val=1.4002
epoch  19/10

In [58]:
# ── STEP 2: d=8, readout=z ────────────────────────────────────────────────────
model_d8, history_d8, out_dim_d8 = train_vqc_c3_optimized(d=8, cfg=CFG)
evaluate_and_save_c3(model_d8, d=8, history=history_d8,
                      topology=CFG['topology'], readout_selector=CFG['readout_selector'])

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


Subsample: 500 sample train
d= 8 | topology=ring | readout=z | output_dim=8 | n_weights=16

d=8 | topology=ring | readout=z | batch=4 | epochs=100 | patience=20
----------------------------------------------------------------------
epoch   1/100 | train=1.4648 | val=1.4081
epoch   2/100 | train=1.4203 | val=1.4764
epoch   3/100 | train=1.4349 | val=1.4376
epoch   4/100 | train=1.4385 | val=1.4306
epoch   5/100 | train=1.3859 | val=1.4327
epoch   6/100 | train=1.4300 | val=1.5702
epoch   7/100 | train=1.4265 | val=1.4169
epoch   8/100 | train=1.4062 | val=1.4757
epoch   9/100 | train=1.4203 | val=1.4109
epoch  10/100 | train=1.4158 | val=1.4171
epoch  11/100 | train=1.4206 | val=1.4470
epoch  12/100 | train=1.4029 | val=1.4215
epoch  13/100 | train=1.3923 | val=1.3907
epoch  14/100 | train=1.4032 | val=1.3860
epoch  15/100 | train=1.4167 | val=1.4264
epoch  16/100 | train=1.3980 | val=1.3971
epoch  17/100 | train=1.3879 | val=1.4051
epoch  18/100 | train=1.4092 | val=1.4125
epoch  19/10

In [59]:
# ── STEP 3: d=16, readout=x_y_z ──────────────────────────────────────────────
# Prima benchmark
CFG['readout_selector'] = 'x_y_z'
CFG['batch_size']       = 8
benchmark_batch_real(16, CFG)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


d=16 | topology=ring | readout=x_y_z | output_dim=24 | n_weights=32
d=16 | tempo medio per batch=0.15s
Stima per epoca (4000 campioni): 153s = 2.6 min


0.15338318119902397

In [60]:
model_d16, history_d16, _ = train_vqc_c3_optimized(d=16, cfg=CFG)
evaluate_and_save_c3(model_d16, d=16, history=history_d16,
                      topology=CFG['topology'], readout_selector=CFG['readout_selector'])

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


Subsample: 500 sample train
d=16 | topology=ring | readout=x_y_z | output_dim=24 | n_weights=32

d=16 | topology=ring | readout=x_y_z | batch=8 | epochs=100 | patience=20
----------------------------------------------------------------------
epoch   1/100 | train=1.4248 | val=1.4095
epoch   2/100 | train=1.3825 | val=1.4256
epoch   3/100 | train=1.4275 | val=1.4256
epoch   4/100 | train=1.4312 | val=1.4443
epoch   5/100 | train=1.4026 | val=1.4882
epoch   6/100 | train=1.4114 | val=1.4009
epoch   7/100 | train=1.4139 | val=1.3536
epoch   8/100 | train=1.4343 | val=1.4290
epoch   9/100 | train=1.4193 | val=1.4337
epoch  10/100 | train=1.3870 | val=1.4050
epoch  11/100 | train=1.4142 | val=1.3925
epoch  12/100 | train=1.3813 | val=1.3994
epoch  13/100 | train=1.4113 | val=1.4292
epoch  14/100 | train=1.4021 | val=1.3501
epoch  15/100 | train=1.4177 | val=1.4249


KeyboardInterrupt: 

In [ ]:
# ── STEP 4: d=16, readout=x_y_z_pair_nn ──────────────────────────────────────
CFG['readout_selector'] = 'x_y_z_pair_nn'
benchmark_batch_real(16, CFG)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


d=16 | topology=ring | readout=x_y_z_pair_nn | output_dim=31 | n_weights=32
d=16 | tempo medio per batch=0.20s
Stima per epoca (4000 campioni): 199s = 3.3 min


0.19889182400002028

In [ ]:
model_d16_nn, history_d16_nn, _ = train_vqc_c3_optimized(d=16, cfg=CFG)
evaluate_and_save_c3(model_d16_nn, d=16, history=history_d16_nn,
                      topology=CFG['topology'], readout_selector=CFG['readout_selector'])

In [ ]:
# ── STEP 5 (opzionale): d=32 solo se i test precedenti sono sostenibili ───────
CFG['readout_selector'] = 'x_y_z_pair_nn'
CFG['reps']             = 2
CFG['batch_size']       = 8
CFG['spsa_batch_size']  = 2

t = benchmark_batch_real(32, CFG)
if t < 30:
    model_d32, history_d32, _ = train_vqc_c3_optimized(d=32, cfg=CFG)
    evaluate_and_save_c3(model_d32, d=32, history=history_d32,
                          topology=CFG['topology'], readout_selector=CFG['readout_selector'])
else:
    print(f'Batch troppo lento ({t:.1f}s) — salta d=32 o riduci ulteriormente la config.')

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


d=32 | topology=ring | readout=x_y_z_pair_nn | output_dim=31 | n_weights=96


No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


d=32 | tempo medio per batch=0.42s
Stima per epoca (4000 campioni): 421s = 7.0 min
Subsample: 500 sample train
d=32 | topology=ring | readout=x_y_z_pair_nn | output_dim=31 | n_weights=96

d=32 | topology=ring | readout=x_y_z_pair_nn | batch=8 | epochs=100 | patience=20
----------------------------------------------------------------------


KeyboardInterrupt: 

## 12. Riesegui tutto con topology='ladder'

In [ ]:
CFG['topology']          = 'ladder'
CFG['readout_selector']  = 'z'
CFG['batch_size']        = 4
CFG['reps']              = 1

results_ladder = {}
for d in [4, 8]:
    model, history, _ = train_vqc_c3_optimized(d=d, cfg=CFG)
    evaluate_and_save_c3(model, d=d, history=history,
                          topology='ladder', readout_selector=CFG['readout_selector'])
    results_ladder[d] = (model, history)